**Background Subtraction**

In [5]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

MOG2_subtractor = cv2.createBackgroundSubtractorMOG2(detectShadows=True, varThreshold=10)
bg_subtractor = MOG2_subtractor
camera = cv2.VideoCapture(r"car.mp4")

if not camera.isOpened():
    print("Error: Could not open video.")
    exit()

while True:
    ret, frame = camera.read()

    if not ret:
        print("End of video.")
        break

    frame = cv2.resize(frame, (780, 480))
    foreground_mask = bg_subtractor.apply(frame)
    _, threshold = cv2.threshold(foreground_mask, 120, 255, cv2.THRESH_BINARY)
    dilated = cv2.dilate(threshold, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)), iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        if cv2.contourArea(contour) > 50:
            x, y, w, h = cv2.boundingRect(contour)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imshow("Foreground Mask", foreground_mask)
    cv2.imshow("Threshold", threshold)
    cv2.imshow("Detection", frame)

    key = cv2.waitKey(0) & 0xFF
    if key == 27 or key == ord('q'):
        print("Exiting...")
        break

camera.release()
cv2.destroyAllWindows()


Exiting...


**Averaging**

In [4]:
alpha = 0.01
fps = 20.0
frame_size = (640, 480)
cap = cv2.VideoCapture(r"car.mp4")
bg_model = None
while True:
    ret, frame = cap.read()
    if not ret:
        print("End of video.")
        break
    frame = cv2.resize(frame, frame_size)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    if bg_model is None:
        bg_model = gray.astype("float")
        continue
    cv2.accumulateWeighted(gray, bg_model, alpha)
    bg_diff = cv2.absdiff(gray, cv2.convertScaleAbs(bg_model))
    _, mask = cv2.threshold(bg_diff, 25, 255, cv2.THRESH_BINARY)
    cv2.imshow("Frame", frame)
    cv2.imshow("Background", cv2.convertScaleAbs(bg_model))
    cv2.imshow("Foreground Mask", mask)
    key = cv2.waitKey(0) & 0xFF

    if key == ord('q'):
        print("Exiting...")
        break

Exiting...


**Scene Change Detection**

In [11]:
threshold = 0.5
cap = cv2.VideoCapture(r"spurs.mp4")
frame_count = 0
scene_count = 0
prev_frame = None
output_folder="spurs"

def calculate_histogram_difference(frame1, frame2):
    hist1 = cv2.calcHist([frame1], [0], None, [256], [0, 256])
    hist2 = cv2.calcHist([frame2], [0], None, [256], [0, 256])
    cv2.normalize(hist1, hist1, 0, 1, cv2.NORM_MINMAX)
    cv2.normalize(hist2, hist2, 0, 1, cv2.NORM_MINMAX)
    diff = cv2.compareHist(hist1, hist2, cv2.HISTCMP_CORREL)
    return diff

while True:
    ret, frame = cap.read()
    if not ret:
        print("End of video.")
        break

    frame_count += 1
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if prev_frame is not None:
        diff = calculate_histogram_difference(prev_frame, gray)
        if diff < threshold:
            scene_count += 1
            print(f"Scene Change Detected at frame {frame_count} (Difference: {diff:.2f})")
            cv2.imwrite(os.path.join(output_folder, f"scene_{scene_count}.png"), frame)
            cv2.putText(frame, f"Diff: {diff:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    prev_frame = gray
    cv2.imshow("Frame", frame)

    if cv2.waitKey(0) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Total Scenes Detected: {scene_count}")

Scene Change Detected at frame 21 (Difference: 0.36)
Scene Change Detected at frame 122 (Difference: 0.07)
Scene Change Detected at frame 377 (Difference: -0.28)
Scene Change Detected at frame 591 (Difference: 0.01)
Scene Change Detected at frame 668 (Difference: -0.08)
Scene Change Detected at frame 802 (Difference: -0.11)
End of video.
Total Scenes Detected: 6
